# 4 — Sliding window : réutiliser le travail


## Fil conducteur

Le chapitre 3 déplaçait des indices pour éliminer des candidats. Une fenêtre
utilise aussi deux bornes, mais conserve en plus l'état d'une portion contiguë.
Nous partirons d'un recalcul naïf, identifierons le travail répété, puis
réutiliserons la somme ou les fréquences de la fenêtre précédente.




*Retirer l’élément sortant et ajouter l’entrant évite de recalculer toute la somme.*


## 1. Le problème : une portion contiguë
Une fenêtre est une portion **consécutive** de la séquence. Dans `[2,1,5,1,3,2]`,
`[1,5,1]` est une fenêtre ; `[2,5,3]` ne l'est pas. Ne confonds pas sous-tableau
contigu et sous-séquence qui peut sauter des éléments.
Pour une taille k, il existe n−k+1 fenêtres si 1 ≤ k ≤ n.

### Exemple illustré — une portion contiguë ne saute aucune case

```text
liste :    [2][1][5][1][3]
fenêtre :     └──────┘       = [1, 5, 1]
```

Les valeurs 1, 5 et 1 se suivent dans la liste. `[2, 5, 3]` ne serait pas une
fenêtre, car elle saute des cases.


## 2. Version naïve et coût réel
Pour chaque position de départ, additionner les k éléments coûte O(k).
Le total est O((n−k+1)k). La borne O(nk) est pratique ; si k est fixé à 3,
le coût est **O(n)**, pas O(n²). Une borne quadratique devient pertinente si k
varie avec n (par exemple k proche de n/2). L'intérêt de la fenêtre glissante
est de ne plus refaire k additions à chaque déplacement.

### Exemple illustré — le recalcul répète le centre

```text
[2 + 1 + 5] = 8
    [1 + 5 + 1] = 7
         ↑   ↑
      recalculés
```

Deux fenêtres voisines partagent `1` et `5`. La version naïve les additionne de
nouveau au lieu de réutiliser la somme précédente.


## 3. Fenêtre fixe : retirer puis ajouter
Première somme : 2+1+5=8. Pour avancer d'une case, 2 sort et 1 entre : 8−2+1=7.
Puis 1 sort et 3 entre : 7−1+3=9. L'invariant est : « la somme conservée est
celle de la fenêtre courante ». On conserve aussi la meilleure somme rencontrée.
Initialise ce maximum avec la première fenêtre, pas 0 : si tous les nombres
sont négatifs, 0 serait une réponse impossible.

| Départ | Fenêtre | Calcul | Somme | Maximum |
|---|---|---|---:|---:|
| 0 | 2,1,5 | première somme | 8 | 8 |
| 1 | 1,5,1 | 8−2+1 | 7 | 8 |
| 2 | 5,1,3 | 7−1+3 | 9 | 9 |
| 3 | 1,3,2 | 9−5+2 | 6 | 9 |

### Exemple illustré — retirer puis ajouter

```text
ancienne somme = 2 + 1 + 5 = 8
nouvelle somme = 8 - 2 + 1 = 7
                       ↑   ↑
                    sort  entre
```

La taille reste fixe : exactement une valeur quitte la fenêtre et une autre y
entre.




*Deux fenêtres voisines partagent leurs éléments centraux : on retire seulement la sortie et on ajoute l’entrée.*

## 4. Fenêtre variable : une condition à maintenir
On place deux bornes `gauche` et `droite`. On étend à droite ; quand la fenêtre
viole la contrainte, on retire des éléments à gauche jusqu'à rétablir la validité.
Pour la plus longue sous-chaîne sans doublon, un set décrit les caractères présents.
Il faut parfois retirer **plusieurs** caractères : d'où `while`, et pas `if`.
Sur `abba`, l'arrivée du second b demande de retirer a puis le premier b.

### Exemple illustré — agrandir, puis réparer

Sur `abca` sans doublon :

```text
[a b c] + a  → doublon de a
 retirer a   → [b c a] valide
```

La borne droite agrandit la fenêtre. La borne gauche avance seulement jusqu’au
retour de la condition « aucun doublon ».




*La borne droite agrandit la fenêtre ; la borne gauche restaure la condition.*


## 5. Pourquoi un while dans un for peut rester linéaire
Droite avance n fois. Gauche ne recule jamais et avance au plus n fois au total.
On ne compte pas n retraits pour **chaque** tour : ce sont n retraits pour toute
l'exécution. C'est une analyse amortie. Avec le set, le temps est O(n) en moyenne ;
l'espace est O(min(n, taille de l'alphabet)).

### Exemple illustré — les deux bornes avancent sans revenir

| Borne | Déplacements maximaux sur `n` cases |
|---|---:|
| droite | `n` |
| gauche | `n` |
| total | au plus `2n` |

Le `while` intérieur peut tourner plusieurs fois sur une étape, mais `gauche` ne
recule jamais. Les déplacements s’additionnent.




*Image — Au total, les deux bornes font au plus 2n déplacements.*


## 6. Conditions d'application
La contiguïté et un état actualisable sont essentiels. Une somme s'actualise facilement ;
un maximum de fenêtre ne se met pas à jour en retirant simplement le maximum :
il faut une autre structure, par exemple une deque monotone.
Pour « plus petite fenêtre de somme ≥ cible », la version classique exige
**des nombres non négatifs** et une cible positive : ajouter ne baisse pas la somme,
retirer ne l'augmente pas. Avec des négatifs, cette logique peut manquer la réponse.
La fenêtre fixe de somme maximale, elle, accepte les nombres négatifs.

### Exemple illustré — une règle de réduction peut devenir fausse

Pour atteindre une somme au moins égale à 5 :

```text
valeurs positives : [2, 3, 1]  → agrandir augmente la somme
avec un négatif   : [6,-10, 9] → agrandir peut la diminuer
```

La fenêtre variable classique suppose souvent des valeurs positives. Sans cette
condition, « trop petit donc agrandir » n’est plus un raisonnement sûr.




*Image — Vérifier les conditions avant d’utiliser la fenêtre.*


## 7. Justifier les deux formes de fenêtre

Pour une fenêtre fixe, la preuve suit trois étapes. Après la somme initiale,
`courant` est la somme des `k` premiers éléments. Lors d'un déplacement, on retire
exactement l'élément qui sort et on ajoute exactement celui qui entre : l'invariant
« `courant` est la somme de la fenêtre courante » est conservé. Quand la borne
droite atteint la fin, toutes les fenêtres de taille `k` ont été examinées une fois.

Pour une fenêtre variable, il faut une propriété **monotone**. Avec des nombres non
négatifs, étendre à droite ne diminue pas la somme et retirer à gauche ne l'augmente
pas. Après le `while`, la fenêtre respecte à nouveau la contrainte. Les débuts
retirés ne redeviendront pas utiles pour une future borne droite : ils produiraient
une fenêtre plus longue. Cette élimination doit être redémontrée pour chaque nouveau
problème ; la présence de deux indices ne suffit pas.

Exemple pour une somme cible 7 :

| droite | valeur ajoutée | somme avant réduction | réductions utiles | meilleur |
|---:|---:|---:|---|---:|
| 0 | 2 | 2 | aucune | — |
| 1 | 3 | 5 | aucune | — |
| 2 | 1 | 6 | aucune | — |
| 3 | 2 | 8 | longueur 4, retire 2 | 4 |
| 4 | 4 | 10 | longueurs 4 puis 3 | 3 |
| 5 | 3 | 9 | longueurs 3 puis 2 | 2 |

La correction repose donc sur deux arguments distincts : la fenêtre conservée
représente fidèlement l'état, et chaque candidat éliminé peut l'être sans perdre
une meilleure réponse.

### Exemple illustré — deux invariants différents

| Forme | Ce qui reste vrai pendant la boucle |
|---|---|
| fenêtre fixe | elle contient exactement `k` éléments et la somme stockée est la sienne |
| fenêtre variable | après réduction, elle respecte de nouveau la contrainte |

La preuve doit nommer la propriété réellement maintenue par la forme choisie.


## Deux solutions comparables

Observe les indices de sortie et d’entrée, puis compte les opérations.


In [1]:
def somme_max_naive(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError("k doit être entre 1 et n")
    return max(sum(nums[i:i+k]) for i in range(len(nums)-k+1))

def somme_max_fenetre(nums, k, trace=False):
    if not 1 <= k <= len(nums):
        raise ValueError("k doit être entre 1 et n")
    courant = sum(nums[i] for i in range(k))
    meilleur = courant
    if trace:
        print(0, courant, meilleur)
    for droite in range(k, len(nums)):
        courant += nums[droite] - nums[droite-k]
        meilleur = max(meilleur, courant)
        if trace:
            print(droite-k+1, courant, meilleur)
    return meilleur
assert somme_max_fenetre([2,1,5,1,3,2], 3, True) == 9
assert somme_max_fenetre([-8,-2,-3], 2) == -5


0 8 8
1 7 8
2 9 9
3 6 9


La version naïve crée aussi une copie temporaire de k éléments avec la tranche : O(k) en espace. La version optimisée évite ces copies : O(n) temps, O(1) espace supplémentaire, en supposant les opérations arithmétiques constantes.


## Fenêtre variable sans doublon

Lis chaque ligne comme une règle de maintien du set.


In [2]:
def longueur_sans_doublon(texte, trace=False):
    presents = set()
    gauche = 0
    meilleur = 0
    for droite, caractere in enumerate(texte):
        while caractere in presents:
            presents.remove(texte[gauche])
            gauche += 1
        presents.add(caractere)
        meilleur = max(meilleur, droite-gauche+1)
        if trace:
            print(gauche, droite, texte[gauche:droite+1], meilleur)
    return meilleur
assert longueur_sans_doublon("abba", True) == 2
assert longueur_sans_doublon("") == 0
assert longueur_sans_doublon("abcabcbb") == 3


0 0 a 1
0 1 ab 2
2 2 b 2
2 3 ba 2


### Trace visuelle d’une fenêtre variable sur `abba`

| Droite | Caractère reçu | Action sur la gauche | Fenêtre valide après action | Meilleure longueur |
|---:|:---:|---|---|---:|
| 0 | `a` | aucune | `a` | 1 |
| 1 | `b` | aucune | `ab` | 2 |
| 2 | `b` | retirer `a`, puis l’ancien `b` | `b` | 2 |
| 3 | `a` | aucune | `ba` | 2 |

À l’étape 2, retirer seulement `a` ne suffit pas : le nouveau `b` serait encore
en double. C’est pourquoi la réduction utilise `while` et non un simple `if`.

Après le while et l’ajout, le set contient exactement les caractères d’une fenêtre sans répétition. Les tranches utilisées uniquement pour afficher la trace peuvent ajouter du coût ; la version sans trace reste linéaire en moyenne.


## Exercices différenciés

Essaie chaque exercice avant de consulter le corrigé. Pour lancer les tests d’un exercice, remplace son `pass`, puis enlève le `#` devant les appels de test. Les cellules incomplètes restent exécutables.


### Palier A — Sommes de toutes les fenêtres

Renvoie la liste des sommes des fenêtres de taille k. Refuse k invalide avec `ValueError`.

<details><summary>Indice progressif</summary>

Calcule une première somme ; ajoute-la à la sortie ; actualise-la à chaque déplacement.

</details>


In [3]:
def sommes_fenetres(nums, k):
    if not 1 <= k <= len(nums):
        raise ValueError("k invalide")
    courant = sum(nums[i] for i in range(k))
    resultat = [courant]
    for droite in range(k, len(nums)):
        courant += nums[droite] - nums[droite-k]
        resultat.append(courant)
    return resultat


In [4]:
assert sommes_fenetres([2,1,5,1,3,2], 3) == [8,7,9,6]
assert sommes_fenetres([-2,-1], 1) == [-2,-1]
assert sommes_fenetres([4], 1) == [4]


**Pourquoi cette solution fonctionne**

Temps O(n). L’état de calcul est O(1), mais la sortie contient n−k+1 sommes : ne dis pas O(1) pour la mémoire totale.


### Palier B — Plus petite fenêtre atteignant une cible

Nombres non négatifs, cible > 0. Renvoie la longueur minimale de fenêtre de somme ≥ cible ; 0 si aucune. Refuse les entrées hors contrat.

<details><summary>Indice progressif</summary>

Étends à droite, puis réduis à gauche tant que la somme reste suffisante. Enregistre la longueur avant de retirer.

</details>


In [5]:
def longueur_min(nums, cible):
    if cible <= 0 or any(x < 0 for x in nums):
        raise ValueError("cible positive et valeurs non négatives requises")
    gauche, courant = 0, 0
    meilleur = len(nums) + 1
    for droite, valeur in enumerate(nums):
        courant += valeur
        while courant >= cible:
            meilleur = min(meilleur, droite-gauche+1)
            courant -= nums[gauche]
            gauche += 1
    return 0 if meilleur == len(nums)+1 else meilleur


In [6]:
assert longueur_min([2,3,1,2,4,3], 7) == 2
assert longueur_min([1,1], 5) == 0
assert longueur_min([], 5) == 0
assert longueur_min([0,0,5], 5) == 1


**Pourquoi cette solution fonctionne**

Les retraits ne peuvent pas augmenter la somme ; on teste toutes les possibilités de réduction utiles. Chaque borne avance au plus n fois : O(n), espace O(1). Avec [1,-1,5] et cible 5, la version non validée peut s’arrêter avec une longueur 3 et manquer [5].


### Palier C — Au plus k caractères distincts

Renvoie la longueur de la plus longue sous-chaîne ayant au plus k caractères distincts. Pour k=0, renvoie 0 ; k négatif est invalide.

<details><summary>Indice progressif</summary>

Un set ne suffit pas : utilise des fréquences, décrémente au retrait, supprime une clé quand sa fréquence atteint zéro.

</details>


In [7]:
def au_plus_k(texte, k):
    if k < 0:
        raise ValueError("k négatif")
    comptes = {}
    gauche = meilleur = 0
    for droite, c in enumerate(texte):
        comptes[c] = comptes.get(c, 0) + 1
        while len(comptes) > k:
            ancien = texte[gauche]
            comptes[ancien] -= 1
            if comptes[ancien] == 0:
                del comptes[ancien]
            gauche += 1
        meilleur = max(meilleur, droite-gauche+1)
    return meilleur


In [8]:
assert au_plus_k("eceba", 2) == 3
assert au_plus_k("aa", 1) == 2
assert au_plus_k("abc", 0) == 0
assert au_plus_k("", 4) == 0


**Pourquoi cette solution fonctionne**

Le nombre de clés représente exactement le nombre de caractères distincts de la fenêtre. O(n) en moyenne, espace O(min(n,k+1)) pendant l’ajout temporaire ; sortie de taille constante.


## Questions de compréhension

1. Pourquoi un `while` interne ne rend-il pas automatiquement l'algorithme quadratique ?
2. Quelle propriété des nombres non négatifs justifie la fenêtre de somme variable ?
3. Pourquoi faut-il initialiser la meilleure somme avec une vraie fenêtre ?
4. Donne un problème contigu pour lequel retirer l'élément sortant ne suffit pas à
   mettre l'état à jour en O(1).


**Éléments de réponse.** On compte tous les déplacements de gauche sur l'exécution,
au plus n. Avec des valeurs non négatives, étendre ne diminue pas la somme et retirer
ne l'augmente pas. Une initialisation à 0 inventerait une réponse sur des valeurs
négatives. Le maximum d'une fenêtre demande par exemple une structure supplémentaire
quand la valeur maximale sort.


## À retenir et vérifier

Fenêtre fixe : état initial, élément sortant, élément entrant. Fenêtre variable : contrainte, expansion, réduction. Pour justifier O(n), compte les déplacements totaux des bornes. Ne généralise pas une méthode dépendant de la positivité à des nombres négatifs.

**Autoévaluation :** peux-tu expliquer l'état des variables sur un petit exemple,
justifier la condition d'arrêt et donner un cas limite ? Peux-tu distinguer
l'espace de travail de l'espace occupé par les résultats ?

**Débrief en binôme :** présente une première solution correcte, puis un compromis
(vitesse, mémoire, lisibilité). Une optimisation doit préserver le contrat.


**Étape suivante.** Le chapitre 5 quitte les parcours linéaires : il décrit un problème à l'aide d'une version plus petite du même problème.
